# TB 3D Reconstruction — Phase 3 (Colab)

Head B (reconstruction) training. Same pipeline as `code/tb_phase3_kaggle.ipynb`;
only the environment handling differs.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough).

**What this run is.** Head B gets its *own* encoder, initialized from the Phase 2
diagnostic model and then free to adapt, with `features[-2]` skip fusion and
film-like DRR augmentation on half the paired samples. Head A is unaffected
because it keeps its own separate weights — a shared encoder trained jointly was
measured to drop held-out diagnostic AUC from 0.889 to 0.65.

**Checkpoints go to Google Drive** every 200 steps, and the training cell resumes
from the last one. Colab disconnects; re-running that cell continues where it
stopped rather than starting over.

## 0. Install

Colab ships torch and timm; `diffdrr` brings `torchio` with it.

In [ ]:
!pip install -q diffdrr pydicom SimpleITK

## 1. Check the GPU

Stop here if this says CPU — Runtime → Change runtime type → GPU.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise SystemExit('No GPU — set Runtime > Change runtime type > GPU before continuing.')

## 2. Mount Drive

Checkpoints and the exported volume are written here so a disconnect costs at
most 200 steps.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Get the data

Two inputs are needed:

| Input | What it is | Size |
|---|---|---|
| LIDC-IDRI CT cache | 150 resampled training CT volumes, pre-cached as `.pt` | ~2 GB |
| Phase 2 encoder | the diagnostic checkpoint Head B initializes from | 14 MB |

Both already exist as private Kaggle datasets, so the fast path is the Kaggle
API. Add your Kaggle token as a Colab secret named `KAGGLE_TOKEN`
(key icon in the left sidebar, "Notebook access" on), or the cell falls back to
fetching CT directly from TCIA, which works but takes ~40 minutes.

The cache deliberately holds only the 150 **training** series — the 12 held-out
series used for evaluation are excluded so a run cannot train on them.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
CT_CACHE_INPUT = None

def _kaggle_token():
    try:
        from google.colab import userdata
        return userdata.get('KAGGLE_TOKEN')
    except Exception:
        return os.environ.get('KAGGLE_TOKEN')

token = _kaggle_token()
if token:
    kdir = Path.home() / '.kaggle'
    kdir.mkdir(exist_ok=True)
    # Accepts either credential format: the newer KGAT_... access token, or the
    # classic kaggle.json contents pasted into the same secret.
    token = token.strip()
    if token.startswith('{'):
        cred = kdir / 'kaggle.json'
    else:
        cred = kdir / 'access_token'
    cred.write_text(token)
    cred.chmod(0o600)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

    for slug, dest in [('nithishravikkumar/lidc-ct-cache-2p5mm', DATA_DIR / 'ct_cache'),
                       ('nithishravikkumar/tb-phase2-encoder', DATA_DIR / 'phase2')]:
        if dest.exists() and any(dest.iterdir()):
            print(f'{dest.name}: already present, skipping download')
            continue
        print(f'Downloading {slug} ...')
        subprocess.run(['kaggle', 'datasets', 'download', '-d', slug,
                        '-p', str(dest), '--unzip'], check=True)
    CT_CACHE_INPUT = DATA_DIR / 'ct_cache'
    print(f'CT cache: {len(list(CT_CACHE_INPUT.glob("*.pt")))} series')
else:
    print('No KAGGLE_TOKEN secret found — CT will be fetched from TCIA below '
          '(~40 min), and you must supply the Phase 2 encoder yourself.')

### 3b. TCIA fallback

Runs only when the Kaggle path above didn't provide the cache.

In [ ]:
import errno
import json
import shutil
import time
import zipfile
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

API = "https://services.cancerimagingarchive.net/nbia-api/services/v1"
LIDC_DIR = Path("/content/data/lidc-idri/dicom")

# 150 series measured ~11GB locally (verified) -> ~75MB/series average.
# Colab's disk has a fixed quota (~70-100GB on a standard runtime); a first attempt at 400
# series (~29GB projected) filled the disk mid-fetch and crashed the whole
# session (repeated ENOSPC retries, then the crash corrupted the notebook's
# own output badly enough that Kaggle's post-run HTML export failed too —
# that confusing secondary error is a symptom of this, not a separate bug).
# 150 keeps real headroom under a 20GB quota; MIN_FREE_GB stops early with a
# clear message instead of retry-looping into the same failure again.
N_SERIES = 150
MIN_FREE_GB = 2.0

def fetch_json(url):
    with urlopen(url, timeout=30) as resp:
        return json.loads(resp.read())

def fetch_series(series_uid, dst_dir):
    dst_dir.mkdir(parents=True, exist_ok=True)
    tmp_zip = dst_dir.parent / f"_tmp_{series_uid}.zip"
    with urlopen(f"{API}/getImage?SeriesInstanceUID={series_uid}", timeout=180) as resp, \
         open(tmp_zip, "wb") as f:
        shutil.copyfileobj(resp, f)
    with zipfile.ZipFile(tmp_zip) as zf:
        zf.extractall(dst_dir)
    tmp_zip.unlink()
    return len(list(dst_dir.glob("*.dcm")))

if CT_CACHE_INPUT is not None:
    ct_series_dirs = sorted(CT_CACHE_INPUT.glob("*.pt"))
    print(f"Using prebuilt CT cache: {len(ct_series_dirs)} series from {CT_CACHE_INPUT}")
else:
    print("Querying LIDC-IDRI CT series list...")
    all_ct_series = fetch_json(f"{API}/getSeries?Collection=LIDC-IDRI&Modality=CT")
    print(f"{len(all_ct_series)} CT series available; fetching first {N_SERIES}")

    ok, failed = 0, []
    for i, s in enumerate(all_ct_series[:N_SERIES], 1):
        free_gb = shutil.disk_usage("/content").free / 1e9
        if free_gb < MIN_FREE_GB:
            print(f"[{i}/{N_SERIES}] STOPPING — only {free_gb:.1f}GB free "
                  f"(< {MIN_FREE_GB}GB safety margin). {ok} series fetched so far; "
                  "continuing with what's already on disk rather than crashing the session.")
            break

        uid = s["SeriesInstanceUID"]
        dst = LIDC_DIR / uid
        if dst.exists() and len(list(dst.glob("*.dcm"))) > 0:
            ok += 1
            continue
        for attempt in range(3):
            try:
                n = fetch_series(uid, dst)
                print(f"[{i}/{N_SERIES}] ok ({n} files)")
                ok += 1
                break
            except OSError as e:
                if e.errno == errno.ENOSPC:
                    print(f"[{i}/{N_SERIES}] disk full — not retrying, stopping fetch loop.")
                    shutil.rmtree(dst, ignore_errors=True)
                    failed.append(uid)
                    break
                print(f"[{i}/{N_SERIES}] attempt {attempt+1} failed: {e}")
                time.sleep(2)
            except (URLError, zipfile.BadZipFile) as e:
                print(f"[{i}/{N_SERIES}] attempt {attempt+1} failed: {e}")
                time.sleep(2)
        else:
            failed.append(uid)

    print(f"\nDone. ok={ok} failed={len(failed)} / {N_SERIES}")
    ct_series_dirs = sorted(LIDC_DIR.glob("*"))


## 4. Config

In [ ]:
# ── Optional CXR paths ───────────────────────────────────────────────────────
# Only used to pick a real chest X-ray for the exported sample reconstruction.
# The shape-induction term is off (see SHAPE_INDUCTION_WEIGHT below), so leave
# these pointing nowhere if you haven't put the datasets on Drive — the export
# falls back to a synthetic input.
TBX11K = '/content/drive/MyDrive/tb_datasets/tbx11k'  # optional; only for the export sample
RAHMAN = '/content/drive/MyDrive/tb_datasets/tb-rahman'  # optional; only for the export sample

# ── Model / training ──────────────────────────────────────────────────────────
IMAGE_SIZE   = 320
VOLUME_SIZE  = 128       # proposal's §9 target — do not promise 256^3
RESAMPLE_MM  = 2.5       # CT resample spacing -> ~128 voxels/axis for a typical chest FOV
BACKBONE     = 'efficientnet_b0'
LR           = 1e-4
STEPS        = 8000      # Raised from 2000 after diagnosing *why* that run's
                          # reconstruction quality was poor: (1) ReconHead's softplus
                          # output layer had its bias barely move in 2000 steps (measured
                          # -0.095, needs ~-3.9 to reach the target CT's near-zero
                          # air/background floor) — fixed with a direct -4.0 init below;
                          # (2) load_ct_volume (DICOM read + resample) measured at
                          # ~7-9s/call locally, dwarfing the ~0.3-0.4s model step — most
                          # of a run's wall-clock was disk I/O, not training. Both fixed;
                          # a local 1000-step test at reduced scale (224px/64^3) with
                          # these fixes went from 0% near-zero predictions (every prior
                          # checkpoint, any step count) to matching the target's own
                          # sparsity within a few points by step 100, and improved every
                          # quality metric (PSNR/SSIM/LPIPS/projection-consistency) vs.
                          # the original 2000-step run. This run is the matched-scale
                          # (320px/128^3) test to confirm that holds at the real
                          # resolution. Checkpoints every 200 steps to OUTPUT_DIR
                          # (unchanged) so a session timeout only costs <=200 steps.
SHAPE_INDUCTION_WEIGHT = 0.0  # 0 disables the unpaired term. A local A/B found it
                              # dominates the paired objective (both realism arms
                              # collapsed to the same poor PSNR/SSIM with it on),
                              # and the full-scale realism run with it on (run 3)
                              # came out worse than no-realism on every metric.
DRR_REALISM      = 'mild'  # film-like augmentation of the paired DRR input (see cell below)
DRR_REALISM_PROB = 0.5     # rest stay clean: a realism-only local run degraded on clean DRRs

# Run 8 — the decisive architecture test. Joint training on a SHARED encoder
# destroys Head A (held-out TBX11K AUC 0.889 -> 0.65; refitting a fresh head
# recovers only to 0.77), and freezing the encoder preserves Head A exactly but
# bounds Head B's patient conditioning (cross-patient r=0.94; both a LR sweep
# and a skip connection failed to move it).
# Here Head B gets its OWN encoder: initialized from Phase 2, then free to
# adapt. Head A is untouched because it keeps its own separate weights, so
# nothing this run does can drift it. Costs a second encoder (~5.3M params).
FREEZE_ENCODER = False  # run 8: Head B's own encoder, free to adapt

INIT_FROM = next(iter(Path('/content/data').rglob('phase2_encoder.pt')), None)
INIT_FROM = str(INIT_FROM) if INIT_FROM else None
RECON_SKIP = True          # fuse features[-2] into the decoder
print(f'Phase 2 encoder: {INIT_FROM}')
BATCH_SIZE   = 1          # DRR generation is per-volume; see readme note on batching
SEED         = 42
OUTPUT_DIR   = Path('/content/drive/MyDrive/tb_outputs/phase3_recon')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 5. Imports

In [ ]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from diffdrr.data import read as diffdrr_read
from diffdrr.drr import DRR

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


## 6. CXR loaders (optional — only for the exported sample)

In [ ]:
import json as _json

_TBX11K_MAP = {
    'healthy': 0, 'sick_but_non-tb': 0,
    'active_tb': 1, 'active&latent_tb': 1,
    'latent_tb': None, 'no_tag': None,
}

def load_tbx11k_paths(root, split='train'):
    root = Path(root)
    img_dir, ann_dir = root / split / 'img', root / split / 'ann'
    paths = []
    for img_path in sorted(img_dir.glob('*.png')):
        ann_path = ann_dir / f'{img_path.name}.json'
        if not ann_path.exists():
            continue
        ann = _json.loads(ann_path.read_text())
        tag = ann['tags'][0]['name'] if ann['tags'] else 'no_tag'
        if _TBX11K_MAP.get(tag) is None:
            continue
        paths.append(img_path)
    return paths

def load_cxr_paths_recursive(root, exts=('*.png', '*.jpg', '*.jpeg')):
    """Shape induction doesn't need labels or a specific class/folder
    structure — any real CXR works for the re-projection consistency term —
    so this globs recursively rather than assuming one dataset's layout
    (the tawsifurrahman TB dataset uses Normal/Tuberculosis subfolders,
    not the NLM images/images/ convention the local pipeline's raddar
    source uses)."""
    root = Path(root)
    paths = []
    for ext in exts:
        paths += list(root.rglob(ext))
    return sorted(paths)

cxr_paths = []
try:
    cxr_paths += load_tbx11k_paths(TBX11K, 'train')
except Exception as e:
    print('TBX11K load skipped:', e)
try:
    cxr_paths += load_cxr_paths_recursive(RAHMAN)
except Exception as e:
    print('Rahman TB dataset load skipped:', e)

print(f'{len(cxr_paths)} unpaired real CXRs for shape induction')
if not cxr_paths:
    print('WARNING: cxr_paths is empty — shape induction will be SKIPPED entirely, '
          'training will only run the paired-DRR term. Check TBX11K/RAHMAN paths '
          'against the actual mounted structure above before trusting a "full" run.')


## 7. Model — shared encoder, diagnostic head, reconstruction head

In [ ]:
import timm

class SharedEncoder(nn.Module):
    def __init__(self, backbone='efficientnet_b0', pretrained=True):
        super().__init__()
        self.net = timm.create_model(backbone, pretrained=pretrained, features_only=True)
        self._channels = self.net.feature_info.channels()

    @property
    def out_channels(self):
        return self._channels[-1]

    @property
    def skip_channels(self):
        return self._channels[-2] if len(self._channels) > 1 else None

    def forward(self, x):
        return self.net(x)


class DiagnosticHead(nn.Module):
    def __init__(self, in_channels, dropout=0.3):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(in_channels, 1)

    def forward(self, features):
        x = self.gap(features[-1]).flatten(1)
        return self.fc(self.drop(x))


class ReconHead(nn.Module):
    """Back-projection reconstruction head — see code/models/recon_head.py for
    the full design note (channel-to-depth lift + 3D decoder + softplus output,
    the softplus fixed a real zero-gradient bug found verifying this locally)."""

    def __init__(self, in_channels, volume_size=128, lift_depth=8, base_channels=32,
                 use_skip=False, skip_channels=None, skip_lift_channels=16):
        super().__init__()
        self.volume_size = volume_size
        self.lift_depth = lift_depth
        self.lift_channels = base_channels * 2
        self.use_skip = use_skip
        self.skip_lift_channels = skip_lift_channels
        self.to_lift = nn.Conv2d(in_channels, self.lift_channels * lift_depth, kernel_size=1)

        def up_block(c_in, c_out):
            return nn.Sequential(
                nn.Upsample(scale_factor=2, mode='trilinear', align_corners=False),
                nn.Conv3d(c_in, c_out, kernel_size=3, padding=1),
                nn.BatchNorm3d(c_out),
                nn.ReLU(inplace=True),
            )

        self.decoder = nn.Sequential(
            up_block(self.lift_channels, base_channels * 2),
            up_block(base_channels * 2, base_channels),
            up_block(base_channels, base_channels // 2),
            up_block(base_channels // 2, base_channels // 4),
        )
        if use_skip:
            # features[-2] carries 4x the spatial resolution of the deepest map.
            # With a frozen encoder the deepest map alone bottlenecks everything
            # patient-specific (conditioning probe: cross-patient r=0.94).
            self.skip_lift = nn.Conv2d(skip_channels, skip_lift_channels * lift_depth * 2,
                                       kernel_size=1)
            self.skip_merge = nn.Conv3d(base_channels * 2 + skip_lift_channels,
                                        base_channels * 2, kernel_size=3, padding=1)

        self.to_density = nn.Conv3d(base_channels // 4, 1, kernel_size=1)
        # Real CT density is ~50-63% near-zero air/background (measured on
        # held-out LIDC-IDRI). Default init leaves this bias near 0, so
        # softplus(0)=ln(2)~=0.69 is every voxel's starting prediction; a
        # 2000-step run only moved it to -0.095 (needs ~-3.9 for near-zero
        # output) -- confirmed root cause of a mode-collapse to a smoothed,
        # input-invariant density blob. Start near the sparse solution
        # directly instead of asking gradient descent to find it.
        nn.init.constant_(self.to_density.bias, -4.0)

    def forward(self, features):
        x = features[-1]
        b, _, h, w = x.shape
        x = self.to_lift(x)
        x = x.view(b, self.lift_channels, self.lift_depth, h, w)
        x = self.decoder[0](x)
        if self.use_skip:
            sk = features[-2]
            sk = self.skip_lift(sk).view(b, self.skip_lift_channels, self.lift_depth * 2,
                                         sk.shape[-2], sk.shape[-1])
            if sk.shape[-3:] != x.shape[-3:]:
                sk = F.interpolate(sk, size=x.shape[-3:], mode='trilinear', align_corners=False)
            x = self.skip_merge(torch.cat([x, sk], dim=1))
        for block in self.decoder[1:]:
            x = block(x)
        x = self.to_density(x)
        if x.shape[-1] != self.volume_size:
            x = F.interpolate(x, size=(self.volume_size,) * 3, mode='trilinear', align_corners=False)
        return F.softplus(x)  # CT density is physically non-negative — see design note above


class TBDiagnosticModel(nn.Module):
    """Isolation guarantee (proposal §11a): forward() never references recon_head.
    forward_recon() never references head. See recon/firewall_test.py in the source repo."""

    def __init__(self, encoder, head, recon_head=None):
        super().__init__()
        self.encoder = encoder
        self.head = head
        self.recon_head = recon_head

    def forward(self, x):
        return self.head(self.encoder(x))

    def forward_recon(self, x):
        return self.recon_head(self.encoder(x))


def build_model(backbone='efficientnet_b0', pretrained=True, dropout=0.3,
                with_recon=False, volume_size=128, recon_skip=False):
    encoder = SharedEncoder(backbone, pretrained)
    head = DiagnosticHead(encoder.out_channels, dropout)
    recon_head = ReconHead(encoder.out_channels, volume_size=volume_size,
                           use_skip=recon_skip,
                           skip_channels=encoder.skip_channels) if with_recon else None
    return TBDiagnosticModel(encoder, head, recon_head)


## 8. CT loading and DRR generation

In [ ]:
def load_ct_volume(series_dir, resample_mm=RESAMPLE_MM):
    return diffdrr_read(str(series_dir), resample_target=resample_mm)

CT_CACHE_DIR = Path('/content/data/ct_cache_built')
CT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def load_ct_volume_cached(series_dir, resample_mm=RESAMPLE_MM):
    """DICOM read + resample measured at ~7-9s/call locally, dwarfing the
    model's forward+backward (~0.3-0.4s) -- caches the loaded Subject to
    disk (torch.save) so repeat draws of the same series (inevitable with
    random.choice over a fixed 150-series pool) are near-free. Verified
    locally: ~900x speedup on a cache hit."""
    series_dir = Path(series_dir)
    if series_dir.suffix == '.pt':      # prebuilt cache shipped as a dataset
        return torch.load(series_dir, weights_only=False)
    cache_path = CT_CACHE_DIR / f'{series_dir.name}.pt'
    if cache_path.exists():
        return torch.load(cache_path, weights_only=False)
    subject = load_ct_volume(series_dir, resample_mm)
    torch.save(subject, cache_path)
    return subject

def build_drr(subject, height=IMAGE_SIZE, sdd=1020.0, delx=2.0):
    return DRR(subject, sdd=sdd, height=height, delx=delx).to(device)

def random_pose(rotation_deg=5.0, translation_mm=20.0, base_translation_y=850.0):
    rot = torch.deg2rad(torch.empty(1, 3, device=device).uniform_(-rotation_deg, rotation_deg))
    trans = torch.empty(1, 3, device=device).uniform_(-translation_mm, translation_mm)
    trans[:, 1] += base_translation_y
    return rot, trans


## 9. DRR realism augmentation

Verbatim copy of `code/recon/drr_realism.py` (tested in `code/tests/`). Makes a clean DRR look
like a real film (framing crop, tone curve, scatter, blur, edge enhancement, noise) to shrink
the DRR -> real CXR domain gap. Input-only: the CT target is never touched.


In [ ]:
import math

import torch
import torch.nn.functional as F
from torchvision.transforms.functional import gaussian_blur

BODY_THRESHOLD = 0.05

# (low, high) uniform ranges per effect. "mild" vs "strong" mirrors Phase 2's
# --aug-strength dial, where mild beat aggressive texture augmentation.
STRENGTHS = {
    "mild": {
        "margin": (-0.12, 0.05), "gamma": (0.45, 0.9), "scatter": (0.0, 0.15),
        "blur": (0.0, 1.0), "unsharp": (0.0, 0.6), "noise": (0.0, 0.02),
    },
    "strong": {
        "margin": (-0.20, 0.10), "gamma": (0.3, 1.1), "scatter": (0.0, 0.30),
        "blur": (0.0, 2.0), "unsharp": (0.0, 1.2), "noise": (0.0, 0.05),
    },
}


def _uniform(lo_hi, n, generator):
    lo, hi = lo_hi
    return lo + (hi - lo) * torch.rand(n, generator=generator)


def _blur(img: torch.Tensor, sigma: float) -> torch.Tensor:
    if sigma <= 0:
        return img
    k = 2 * math.ceil(3 * sigma) + 1
    return gaussian_blur(img, kernel_size=[k, k], sigma=[sigma, sigma])


def crop_to_body(x: torch.Tensor, margin: torch.Tensor) -> torch.Tensor:
    """Square crop around each sample's body bounding box, expanded by
    margin * box size on each side, resized back to the input resolution."""
    out = x.clone()
    _, _, H, W = x.shape
    for i in range(x.shape[0]):
        mask = x[i, 0] > BODY_THRESHOLD
        if not mask.any():
            continue
        rows = torch.nonzero(mask.any(dim=1)).squeeze(1)
        cols = torch.nonzero(mask.any(dim=0)).squeeze(1)
        r0, r1, c0, c1 = rows[0].item(), rows[-1].item() + 1, cols[0].item(), cols[-1].item() + 1
        side = max(r1 - r0, c1 - c0) * (1 + 2 * margin[i].item())
        side = int(min(max(side, 1), H, W))
        cy, cx = (r0 + r1) / 2, (c0 + c1) / 2
        top = int(min(max(round(cy - side / 2), 0), H - side))
        left = int(min(max(round(cx - side / 2), 0), W - side))
        crop = x[i:i + 1, :, top:top + side, left:left + side]
        out[i:i + 1] = F.interpolate(crop, size=(H, W), mode="bilinear", align_corners=False)
    return out


def realistic_drr(x: torch.Tensor, strength: str = "mild",
                  generator: torch.Generator | None = None, prob: float = 1.0) -> torch.Tensor:
    """prob: per-sample chance of applying the effect; the rest pass through
    clean, so the model keeps seeing clean DRRs during training."""
    if strength == "off":
        return x
    p = STRENGTHS[strength]
    n = x.shape[0]
    apply = torch.rand(n, generator=generator) < prob
    draw = {name: _uniform(rng, n, generator) for name, rng in p.items()}
    _, _, H, _ = x.shape

    x_clean = x
    x = crop_to_body(x, draw["margin"])
    out = []
    for i in range(n):
        xi = x[i:i + 1].clamp(0, 1) ** draw["gamma"][i].item()
        a = draw["scatter"][i].item()
        xi = (1 - a) * xi + a * _blur(xi, H / 8)
        xi = _blur(xi, draw["blur"][i].item())
        xi = xi + draw["unsharp"][i].item() * (xi - _blur(xi, max(1.0, H / 100)))
        noise = torch.randn(xi.shape, generator=generator).to(xi.device)
        xi = xi + draw["noise"][i].item() * noise
        out.append(xi.clamp(0, 1))
    return torch.where(apply.view(-1, 1, 1, 1).to(x.device), torch.cat(out), x_clean)


## 10. Training steps

In [ ]:
def drr_to_model_input(drr_img, image_size=IMAGE_SIZE, realism='off', realism_prob=1.0):
    x = drr_img
    x = (x - x.amin(dim=(2, 3), keepdim=True)) / (
        x.amax(dim=(2, 3), keepdim=True) - x.amin(dim=(2, 3), keepdim=True) + 1e-8)
    x = realistic_drr(x, realism, prob=realism_prob)
    x = F.interpolate(x, size=(image_size, image_size), mode='bilinear', align_corners=False)
    x = x.repeat(1, 3, 1, 1)
    return (x - _MEAN.to(x.device)) / _STD.to(x.device)

def real_cxr_to_model_input(path, image_size=IMAGE_SIZE):
    img = Image.open(path).convert('RGB').resize((image_size, image_size))
    x = torch.from_numpy(np.array(img)).float().permute(2, 0, 1) / 255.0
    return ((x - _MEAN[0]) / _STD[0]).unsqueeze(0)

def paired_step(model, ct_series):
    subject = load_ct_volume_cached(ct_series)
    drr = build_drr(subject)
    rot, trans = random_pose()
    drr_img = drr(rot, trans, parameterization='euler_angles', convention='ZXY')

    x = drr_to_model_input(drr_img, realism=DRR_REALISM, realism_prob=DRR_REALISM_PROB)
    pred_volume = model.forward_recon(x)

    target = subject.density.data.to(device).squeeze(0)
    target = F.interpolate(target[None, None], size=(VOLUME_SIZE,) * 3,
                           mode='trilinear', align_corners=False)
    return F.mse_loss(pred_volume, target)

def shape_induction_step(model, cxr_path, canonical_drr, canonical_volume_shape):
    x = real_cxr_to_model_input(cxr_path).to(device)
    pred_volume = model.forward_recon(x)

    resized = F.interpolate(pred_volume, size=canonical_volume_shape,
                            mode='trilinear', align_corners=False)
    canonical_drr.density = resized.squeeze(0).squeeze(0)
    rot, trans = random_pose(rotation_deg=0.0, translation_mm=0.0)
    reprojected = canonical_drr(rot, trans, parameterization='euler_angles', convention='ZXY')

    def _norm(t):
        return (t - t.mean()) / (t.std() + 1e-8)
    return F.mse_loss(_norm(reprojected), _norm(F.interpolate(
        x[:, :1], size=reprojected.shape[-2:], mode='bilinear', align_corners=False)))


@torch.no_grad()
def sparsity_diagnostic(model, ct_series):
    """Prints the fraction of predicted voxels below 0.02 density vs. the
    target's own -- real CT is ~50-63% near-zero air/background; a healthy
    fit should track that, not stay at 0% (the mode-collapse signature this
    whole fix was built to resolve). See code/readme.md's Phase 3 section."""
    was_training = model.training
    model.eval()
    subject = load_ct_volume_cached(ct_series)
    drr = build_drr(subject)
    rot, trans = random_pose(rotation_deg=0.0, translation_mm=0.0)
    img = drr(rot, trans, parameterization='euler_angles', convention='ZXY')
    pred = model.forward_recon(drr_to_model_input(img))
    target = F.interpolate(subject.density.data.to(device)[None], size=(VOLUME_SIZE,) * 3,
                           mode='trilinear', align_corners=False)
    frac_pred = (pred < 0.02).float().mean().item()
    frac_target = (target < 0.02).float().mean().item()
    print(f'  [diagnostic] pred min={pred.min().item():.4f} mean={pred.mean().item():.4f} '
          f'frac<0.02={frac_pred:.1%}   target frac<0.02={frac_target:.1%}')
    if was_training:
        model.train()


## 11. Train

Auto-checkpoints every 200 steps to `OUTPUT_DIR` so a Kaggle session timeout
doesn't lose progress — re-run this cell after restarting to resume (loads
the last checkpoint if present).

In [ ]:
ct_series_dirs = (sorted(CT_CACHE_INPUT.glob('*.pt')) if CT_CACHE_INPUT is not None
                  else sorted(LIDC_DIR.glob('*')))
print(f'CT series for training: {len(ct_series_dirs)}')

import time

# pretrained=False when INIT_FROM supplies the encoder: the ImageNet weights
# would be overwritten immediately anyway, and fetching them needs internet
# (HuggingFace), which an unverified account's kernels do not have.
model = build_model(BACKBONE, pretrained=(INIT_FROM is None), with_recon=True,
                    volume_size=VOLUME_SIZE, recon_skip=RECON_SKIP).to(device)

if INIT_FROM:
    _sd = torch.load(INIT_FROM, map_location=device, weights_only=False)['model']
    _missing, _unexpected = model.load_state_dict(_sd, strict=False)
    print(f'Initialized encoder from {INIT_FROM} ({len(_sd)} keys, {len(_unexpected)} unused)')

if FREEZE_ENCODER:
    for _p in model.encoder.parameters():
        _p.requires_grad_(False)
    model.encoder.eval()
    # Stay in eval mode through model.train(), or BatchNorm running stats drift
    # on DRR inputs and Head A changes anyway, frozen weights or not.
    model.encoder.train = lambda mode=True: model.encoder
    trainable = list(model.recon_head.parameters())
else:
    trainable = list(model.encoder.parameters()) + list(model.recon_head.parameters())
print(f'Trainable tensors: {len(trainable)}'
      f" ({'Head B only, encoder frozen' if FREEZE_ENCODER else 'encoder + Head B'})")
optimizer = torch.optim.AdamW(trainable, lr=LR)

ckpt_path = OUTPUT_DIR / 'latest.pt'
start_step = 1
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_step = ckpt['step'] + 1
    print(f'Resumed from step {ckpt["step"]}')

canonical_subject = load_ct_volume_cached(ct_series_dirs[0])
canonical_drr = build_drr(canonical_subject)
canonical_volume_shape = tuple(canonical_drr.density.shape)

# One-time cache warm-up: measured ~742s for 162 series locally (~4.6s/series
# avg) -- pays for itself almost immediately once training starts drawing
# the same series repeatedly via random.choice.
print(f'Warming CT cache for {len(ct_series_dirs)} series...'
      if CT_CACHE_INPUT is None else 'Prebuilt CT cache — skipping warm-up')
_t0 = time.time()
for _i, _s in enumerate(ct_series_dirs if CT_CACHE_INPUT is None else [], 1):
    load_ct_volume_cached(_s)
    if _i % 25 == 0:
        print(f'  [{_i}/{len(ct_series_dirs)}] elapsed={time.time()-_t0:.0f}s')
print(f'Cache warm-up done in {time.time()-_t0:.0f}s')

model.train()
for step in range(start_step, STEPS + 1):
    optimizer.zero_grad()

    loss_paired = paired_step(model, random.choice(ct_series_dirs))
    loss = loss_paired
    log = f'step {step}/{STEPS}  paired={loss_paired.item():.4f}'

    if cxr_paths and SHAPE_INDUCTION_WEIGHT > 0:
        loss_shape = shape_induction_step(model, random.choice(cxr_paths),
                                          canonical_drr, canonical_volume_shape)
        loss = loss + SHAPE_INDUCTION_WEIGHT * loss_shape
        log += f'  shape_induction={loss_shape.item():.4f}'

    loss.backward()
    optimizer.step()

    if step % 20 == 0 or step == STEPS:
        print(log + f'  total={loss.item():.4f}')
    if step % 200 == 0 or step == STEPS:
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                   'step': step}, ckpt_path)
        sparsity_diagnostic(model, random.choice(ct_series_dirs))

print('\nDone.')


## 12. Export a sample reconstruction (NRRD)

In [ ]:
import SimpleITK as sitk

SYNTHESIZED_TAG = 'SYNTHESIZED — not for measurement or diagnosis'

def export_volume(volume, out_path, spacing_mm=RESAMPLE_MM):
    arr = np.squeeze(volume.detach().cpu().numpy()).astype(np.float32)
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing((spacing_mm, spacing_mm, spacing_mm))
    img.SetMetaData('description', SYNTHESIZED_TAG)
    sitk.WriteImage(img, str(out_path))
    return out_path

model.eval()
if cxr_paths:
    sample_cxr = random.choice(cxr_paths)
    x = real_cxr_to_model_input(sample_cxr).to(device)
    print(f'Exporting a reconstruction from real CXR: {sample_cxr.name}')
else:
    # Don't let a data-path issue crash the run after training already
    # succeeded and checkpointed — fall back to a synthetic input so the
    # export cell still demonstrates the mechanism runs end to end.
    print('cxr_paths is empty — exporting from a synthetic random input instead '
          'of a real CXR (fix the TBX11K/RAHMAN paths for a real demo).')
    x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
with torch.no_grad():
    volume = model.forward_recon(x)

out = export_volume(volume, OUTPUT_DIR / 'sample_reconstruction.nrrd')
print(f'Exported {out} from input {sample_cxr.name}')
print(SYNTHESIZED_TAG)
